In [1]:
import pandas as pd
import numpy as np

# imputed values
from sklearn.impute import KNNImputer

# datos outliers
from pyod.models.knn import KNN

# Procesado y modelado
from sklearn.model_selection import train_test_split

from sklearn.metrics import mean_squared_error
import lightgbm as lgb

# configuracion warnings
import warnings

warnings.filterwarnings("ignore")

# carga de datos

In [2]:
df_harvest = pd.read_excel("../DB-HISTORICA.xlsx", sheet_name="SEKOYA POP")
df_harvest = df_harvest.loc[
    :, ["FORMATO", "KGHORA", "INDUSTRIAL", "KGHECT", "PesoBayaFIMPRO", "HECTA", "DPC"]
]

# imputacion de valores perdidos

In [3]:
cols_with_missing = ["KGHORA", "INDUSTRIAL", "KGHECT", "PesoBayaFIMPRO", "HECTA", "DPC"]
imputer = KNNImputer(n_neighbors=10, weights="distance")
imputer.fit(df_harvest[cols_with_missing])
df_harvest[cols_with_missing] = imputer.transform(df_harvest[cols_with_missing])

# creacion de variables dummy o tratamiento de variable categoricas

In [4]:
df_formato = pd.get_dummies(df_harvest["FORMATO"])
df_harvest_new = pd.concat(
    [
        df_harvest.loc[:, ["KGHORA", "INDUSTRIAL", "KGHECT", "PesoBayaFIMPRO"]],
        df_formato,
    ],
    axis=1,
)
df_harvest_new.shape

(1269, 8)

# Tratamiento de valores Outliers

In [5]:
clf = KNN(contamination=0.02)
clf.fit(df_harvest_new)
y_pred_clf = clf.predict(df_harvest_new)
df_harvest_new = df_harvest_new[y_pred_clf == 0]
df_harvest_new.shape

(1247, 8)

# division entre train y test nuestro dataset

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    df_harvest_new.drop(columns="KGHORA"),
    df_harvest_new["KGHORA"],
    random_state=123,
    test_size=0.2,
)

In [7]:
hyper_params = {
    "task": "train",
    "boosting_type": "gbdt",
    "objective": "regression",
    "metric": ["l1", "l2"],
    "learning_rate": 0.005,
    "feature_fraction": 1,
    "bagging_fraction": 1,
    "bagging_freq": 30,
    "min_data_in_leaf": 3,
    "verbose": -1,
    "max_depth": 8,
    "num_leaves": 45,
    "max_bin": 512,
    "num_iterations": 100000,
    "n_estimators": 100,
    "n_jobs": 1,
}
gbm = lgb.LGBMRegressor(**hyper_params)
gbm.fit(X_train, y_train, eval_set=[(X_test, y_test)], eval_metric="l1")

,boosting_type,'gbdt'
,num_leaves,45
,max_depth,8
,learning_rate,0.005
,n_estimators,100
,subsample_for_bin,200000
,objective,'regression'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [8]:
y_pred = gbm.predict(X_train, num_iteration=gbm.best_iteration_)
y_pred_test = gbm.predict(X_test, num_iteration=gbm.best_iteration_)

In [9]:
print("The rmse of train is:", round(mean_squared_error(y_pred, y_train) ** 0.5, 5))
print("The rmse of test is:", round(mean_squared_error(y_pred_test, y_test) ** 0.5, 5))

The rmse of train is: 0.0
The rmse of test is: 1.53168


# **PROGRAMACION**


In [10]:
# cargar datos de la programacion
df_programacion = pd.read_excel("../BD_PYO_INPUTS.xlsx", sheet_name="POP")

In [11]:
df_programacion_info = df_programacion.loc[
    :,
    [
        "SEMANA",
        "KEY_DPC",
        "VARIEDAD",
        "FORMATO",
        "KGHECT",
        "HORAS_EFECTIVAS",
        "INDUSTRIAL",
        "TN",
        "HECTA",
        "PesoBayaFIMPRO",
    ],
]

In [12]:
# imputed values
columnames = ["HECTA", "KGHECT", "INDUSTRIAL", "DPC", "PesoBayaFIMPRO", "FORMATO"]
df_programacion = df_programacion[columnames]
df_numeric = df_programacion.select_dtypes(include=["float", "int64"])
df_categorica = df_programacion.select_dtypes(include="object")
imputer = KNNImputer(n_neighbors=10, weights="distance")
imputed = imputer.fit_transform(df_numeric)
df_imputed = pd.DataFrame(
    imputed, columns=["HECTA", "KGHECT", "INDUSTRIAL", "DPC", "PesoBayaFIMPRO"]
)
df_programacion = pd.concat([df_imputed, df_categorica], axis=1)

# Transformar valores dummies
df_formato_prog = pd.get_dummies(df_programacion["FORMATO"])
df_programacion_new = pd.concat(
    [
        df_programacion.loc[:, ["INDUSTRIAL", "KGHECT", "PesoBayaFIMPRO"]],
        df_formato_prog,
    ],
    axis=1,
)

In [13]:
tipo_cosecha = list(df_harvest["FORMATO"].unique())
formato = list(df_programacion["FORMATO"].unique())

In [14]:
for i in tipo_cosecha:
    if i not in formato:
        df_programacion_new[i] = 0

In [15]:
df_programacion_new = df_programacion_new.loc[
    :,
    [
        "INDUSTRIAL",
        "KGHECT",
        "PesoBayaFIMPRO",
        "CLAMSHELL 11 OZ",
        "CLAMSHELL 18 OZ",
        "CLAMSHELL 4.4 OZ",
        "GRANEL",
    ],
]

In [16]:
y_predicho = gbm.predict(X=df_programacion_new)
df_programacion_info["Y_PRED"] = y_predicho
np.savetxt("../pronosticos/pop.csv", df_programacion_info, delimiter=",", fmt="%s")